# Evaluate ASVspoof 5 Track 1

This notebook evaluates ASVspoof 5 Track 1, the stand-alone countermeasure task. It saves a unified `/kaggle/working/asvspoof5/results.pkl` after each model and keeps per-model partial checkpoints so long runs can resume after interruption.

## Dataset structure

ASVspoof 5 audio is 16 kHz FLAC. Track 1 metadata files are space-separated protocol files such as `ASVspoof5.train.tsv`, `ASVspoof5.dev.track_1.tsv`, and `ASVspoof5.eval.track_1.tsv`. Rows contain `SPEAKER_ID FLAC_FILE_NAME SPEAKER_GENDER CODEC CODEC_Q CODEC_SEED ATTACK_TAG ATTACK_LABEL KEY TMP`; `KEY` is the CM label (`bonafide` or `spoof`). The corresponding audio prefixes are `flac_T` for train, `flac_D` for dev, and `flac_E` for eval. Track 2 enrollment/trial files are SASV protocols and are intentionally not used here.

In [ ]:
import gc
import importlib
import io
import json
import os
import pickle
import subprocess
import sys
import tarfile
import time
import base64
from pathlib import Path

import numpy as np
import pandas as pd

# A failed import can leave torch half-initialized in the notebook kernel.
# Clear that stale state before retrying imports in the same session.
_torch_mod = sys.modules.get("torch")
if _torch_mod is not None and not hasattr(_torch_mod, "Tensor"):
    for _name in list(sys.modules):
        if _name == "torch" or _name.startswith("torch."):
            del sys.modules[_name]

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from sklearn.metrics import roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import soundfile as sf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "soundfile"])
    import soundfile as sf

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


def ensure_parent(path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def compute_eer(scores, labels) -> float:
    """Compute EER in percent. Higher score means more bonafide."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1.0 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) * 50.0)


def load_pickle_results(candidates: list[Path]) -> dict:
    """Load the first existing results.pkl candidate."""
    for path in candidates:
        if path.exists():
            print(f"loading existing results: {path}")
            with open(path, "rb") as handle:
                return pickle.load(handle)
    return {}


def save_results_pickle(results: dict, output_path: Path) -> None:
    """Persist the unified dataset results file after each model finishes."""
    slim = {}
    for model_name, result in results.items():
        slim[model_name] = {
            "eer": float(result["eer"]),
            "scores": np.asarray(result["scores"], dtype=np.float64),
            "labels": np.asarray(result["labels"], dtype=np.int64),
        }
    ensure_parent(output_path)
    with open(output_path, "wb") as handle:
        pickle.dump(slim, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"saved {output_path} with models={list(slim)}")


def partial_path(output_dir: Path, model_name: str) -> Path:
    safe = model_name.replace("+", "_").replace(" ", "_").replace("/", "_")
    return output_dir / f"{safe}.partial.npz"


def load_partial(output_dir: Path, model_name: str, fallback_dirs: list[Path] | None = None) -> dict | None:
    path = partial_path(output_dir, model_name)
    if not path.exists():
        for fb_dir in (fallback_dirs or []):
            candidate = partial_path(fb_dir, model_name)
            if candidate.exists():
                path = candidate
                print(f"{model_name}: resuming partial from input dataset: {path}")
                break
    if not path.exists():
        return None
    data = np.load(path, allow_pickle=True)
    return {
        "scores": data["scores"].astype(np.float64).tolist(),
        "labels": data["labels"].astype(np.int64).tolist(),
        "utt_ids": data["utt_ids"].astype(str).tolist(),
    }


def save_partial(output_dir: Path, model_name: str, scores: list, labels: list, utt_ids: list) -> None:
    path = partial_path(output_dir, model_name)
    np.savez(
        ensure_parent(path),
        scores=np.asarray(scores, dtype=np.float64),
        labels=np.asarray(labels, dtype=np.int64),
        utt_ids=np.asarray(utt_ids, dtype=str),
    )


def clear_partial(output_dir: Path, model_name: str) -> None:
    path = partial_path(output_dir, model_name)
    if path.exists():
        path.unlink()


def release_model(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
DATASET_KEY = "asvspoof5"
OUTPUT_DIR = Path("/kaggle/working/asvspoof5")
OUTPUT_PKL = OUTPUT_DIR / "results.pkl"
INPUT_RESULTS = [
    OUTPUT_PKL,
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof5/results.pkl"),
    Path("/kaggle/input/sdd-survey/asvspoof5/results.pkl"),
    Path("results/asvspoof5/results.pkl"),
]

# ASVspoof 5 Track 1 is stand-alone countermeasure evaluation: bonafide vs spoof.
# Use "dev" first for a full-system trial, then "eval" for the official large run.
ASV5_SPLIT = "dev"       # "train", "dev", or "eval"
ASV5_TRACK = "track_1"   # this notebook evaluates Track 1 only
ASV5_SOURCE = "hf_tar"   # "hf_tar", "auto", "local", or "hf_webdataset"
SMOKE_TEST_N = None
HF_DEBUG_N = 30
FORCE_EVAL = {
    "AASIST": False,
    "LFCC+LCNN": False,
    "AASIST3": False,
    "AASIST-L": False,
    "XLS-R+AASIST": False,
    "XLS-R+Nes2Net": False,
}
ENABLED_MODELS = ["AASIST", "AASIST-L", "AASIST3", "LFCC+LCNN", "XLS-R+Nes2Net", "XLS-R+AASIST"]
PARTIAL_SAVE_EVERY = 5000
NUM_WORKERS = 2
RUN_COMPOUND_SSL_MODELS = False
PARTIAL_INPUT_DIRS = [
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/asvspoof5"),
    Path("/kaggle/input/sdd-survey/asvspoof5"),
    Path("/kaggle/input/datasets/minhbhm/sdd-partials-asv5"),
    Path("/kaggle/input/sdd-partials-asv5"),
]


def asv5_prefix_for_split(split: str) -> str:
    return {"train": "T", "dev": "D", "eval": "E"}[split]


def locate_asvspoof5_local() -> dict:
    """Find ASVspoof 5 Track 1 protocol and extracted FLAC directory."""
    slugs = ["asvspoof5", "asvspoof-5", "asvspoof5-data", "asvspoof5-dev", "asvspoof5-eval"]
    roots = [Path("/kaggle/input/datasets")] + [Path("/kaggle/input")]
    proto_names = [
        f"ASVspoof5.{ASV5_SPLIT}.{ASV5_TRACK}.tsv",
        f"{ASV5_SPLIT}.{ASV5_TRACK}.tsv",
        f"{ASV5_SPLIT}.track_1.tsv",
        "ASVspoof5.train.tsv" if ASV5_SPLIT == "train" else "",
        "train.tsv" if ASV5_SPLIT == "train" else "",
    ]
    prefix = asv5_prefix_for_split(ASV5_SPLIT)
    audio_dir_names = [f"flac_{prefix}", f"flac_{ASV5_SPLIT}", ASV5_SPLIT]

    bases = []
    for root in roots:
        if not root.exists():
            continue
        for slug in slugs:
            candidate = root / slug
            if candidate.exists():
                bases.append(candidate)
        bases.extend([p for p in root.glob("*asvspoof*5*") if p.is_dir()])

    seen = set()
    for base in bases:
        if base in seen:
            continue
        seen.add(base)
        proto = None
        for name in proto_names:
            if not name:
                continue
            matches = list(base.rglob(name))
            if matches:
                proto = matches[0]
                break
        if proto is None:
            matches = list(base.rglob(f"*{ASV5_SPLIT}*{ASV5_TRACK}*.tsv"))
            if not matches and ASV5_SPLIT == "train":
                matches = list(base.rglob("*train*.tsv"))
            proto = matches[0] if matches else None

        audio_dir = None
        for name in audio_dir_names:
            matches = [p for p in base.rglob(name) if p.is_dir()]
            if matches:
                audio_dir = matches[0]
                break
        if audio_dir is None:
            for p in base.rglob(f"flac_{prefix}*"):
                if p.is_dir():
                    audio_dir = p
                    break

        if proto and audio_dir:
            return {"base": base, "protocol": proto, "audio_dir": audio_dir}

    raise FileNotFoundError(
        "ASVspoof 5 local files were not found. Expected Track 1 TSV plus extracted flac_T/flac_D/flac_E."
    )


def parse_asvspoof5_track1_protocol(protocol_path: Path, audio_dir: Path) -> pd.DataFrame:
    """Parse ASVspoof 5 Track 1 metadata.

    Official Track 1 rows are space-separated:
    SPEAKER_ID FLAC_FILE_NAME SPEAKER_GENDER CODEC CODEC_Q CODEC_SEED
    ATTACK_TAG ATTACK_LABEL KEY TMP
    KEY is the CM label: bonafide or spoof.
    """
    audio_index = {}
    for path in audio_dir.rglob("*"):
        if path.is_file():
            audio_index[path.name] = str(path)
            audio_index[path.stem] = str(path)

    rows = []
    with open(protocol_path, "r", encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) >= 10:
                speaker, utt_id, gender, codec, codec_q, codec_seed, attack_tag, attack_label, key, tmp = parts[:10]
            elif len(parts) >= 5:
                # Defensive fallback for simplified TSV exports.
                speaker, utt_id, gender, attack_label, key = parts[:5]
                codec, codec_q, codec_seed, attack_tag, tmp = "-", "-", "-", "-", "-"
            else:
                continue
            if key not in ("bonafide", "spoof"):
                continue
            rows.append({
                "speaker": speaker,
                "utt_id": utt_id,
                "gender": gender,
                "codec": "nocodec" if codec == "-" else codec,
                "codec_q": codec_q,
                "codec_seed": codec_seed,
                "attack_tag": "bonafide" if attack_tag == "-" else attack_tag,
                "attack": "bonafide" if attack_label == "bonafide" else attack_label,
                "label": 1 if key == "bonafide" else 0,
                "audio_path": audio_index.get(utt_id) or audio_index.get(f"{utt_id}.flac"),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No Track 1 rows parsed from {protocol_path}")
    df = df[df["audio_path"].notna()].copy()
    missing = ~df["audio_path"].map(lambda p: Path(p).exists())
    if missing.any():
        print(f"warning: dropping {int(missing.sum())} ASVspoof 5 rows with missing audio")
        df = df[~missing].copy()
    return df.reset_index(drop=True)


def build_asvspoof5_local_df() -> pd.DataFrame:
    loc = locate_asvspoof5_local()
    print(loc)
    df = parse_asvspoof5_track1_protocol(loc["protocol"], loc["audio_dir"])
    if SMOKE_TEST_N is not None:
        df = df.iloc[:SMOKE_TEST_N].copy()
    print(df["label"].value_counts().rename({1: "bonafide", 0: "spoof"}))
    print(f"ASVspoof 5 {ASV5_SPLIT} Track 1 rows with audio: {len(df):,}")
    print(df[["utt_id", "codec", "attack_tag", "attack", "label", "audio_path"]].head())
    return df


if ASV5_SOURCE == "local":
    eval_df = build_asvspoof5_local_df()
    RESOLVED_ASV5_SOURCE = "local"
elif ASV5_SOURCE == "hf_webdataset":
    eval_df = None
    RESOLVED_ASV5_SOURCE = "hf_webdataset"
elif ASV5_SOURCE == "hf_tar":
    eval_df = None
    RESOLVED_ASV5_SOURCE = "hf_tar"
elif ASV5_SOURCE == "auto":
    try:
        eval_df = build_asvspoof5_local_df()
        RESOLVED_ASV5_SOURCE = "local"
    except FileNotFoundError as exc:
        print(f"local ASVspoof 5 not found; falling back to Hugging Face tar files: {exc}")
        eval_df = None
        RESOLVED_ASV5_SOURCE = "hf_tar"
else:
    raise ValueError(f"Unknown ASV5_SOURCE={ASV5_SOURCE}")

print(f"resolved ASVspoof 5 source: {RESOLVED_ASV5_SOURCE}")
results = load_pickle_results(INPUT_RESULTS)

In [ ]:
import gc
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np

_torch_mod = sys.modules.get("torch")
if _torch_mod is not None and not hasattr(_torch_mod, "Tensor"):
    for _name in list(sys.modules):
        if _name == "torch" or _name.startswith("torch."):
            del sys.modules[_name]

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset


class SimpleLCNN(nn.Module):
    """Small LFCC+LCNN baseline. This must match the saved checkpoint architecture."""

    def __init__(self, n_lfcc: int = 60):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


LFCC_TRANSFORM = torchaudio.transforms.LFCC(
    sample_rate=16000,
    n_lfcc=60,
    speckwargs={"n_fft": 512, "hop_length": 160, "win_length": 320},
)


def run_shell(command: str) -> None:
    """Notebook-friendly shell runner."""
    print(command)
    rc = os.system(command)
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {command}")


def ensure_aasist_repo(repo_dir: Path = Path("/kaggle/working/aasist")) -> Path:
    """Clone clovaai/aasist only when missing."""
    if not repo_dir.exists():
        run_shell(f"git clone https://github.com/clovaai/aasist.git {repo_dir}")
    repo_str = str(repo_dir)
    if repo_str in sys.path:
        sys.path.remove(repo_str)
    sys.path.insert(0, repo_str)
    return repo_dir


def ensure_aasist3_repo(repo_dir: Path = Path("/kaggle/working/AASIST3")) -> Path:
    """Clone mtuciru/AASIST3 only when missing."""
    if not repo_dir.exists():
        run_shell(f"git clone https://github.com/mtuciru/AASIST3.git {repo_dir}")
    repo_str = str(repo_dir)
    if repo_str in sys.path:
        sys.path.remove(repo_str)
    sys.path.insert(0, repo_str)
    return repo_dir


def load_aasist_model(config_name: str, weight_name: str) -> nn.Module:
    """Load AASIST or AASIST-L from the official repository."""
    repo_dir = ensure_aasist_repo()
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["models"])
        with open(repo_dir / "config" / config_name, "r", encoding="utf-8") as handle:
            cfg = json.load(handle)
        module = importlib.import_module(f"models.{cfg['model_config']['architecture']}")
        model = module.Model(cfg["model_config"]).to(DEVICE)
        state = torch.load(repo_dir / "models" / "weights" / weight_name, map_location=DEVICE)
        model.load_state_dict(state)
        return model.eval()
    finally:
        os.chdir(cwd)


def load_aasist3_model() -> nn.Module:
    """Load AASIST3 from Hugging Face through the AASIST3 repository wrapper."""
    repo_dir = ensure_aasist3_repo()
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["model"])
        from model import aasist3
        return aasist3.from_pretrained("MTUCI/AASIST3").to(DEVICE).eval()
    finally:
        os.chdir(cwd)


SSL_AASIST_REPO_URL = "https://github.com/TakHemlata/SSL_Anti-spoofing.git"
SSL_AASIST_REPO_DIR = Path("/kaggle/working/SSL_Anti-spoofing")
NES2NET_REPO_URL = "https://github.com/Liu-Tianchi/Nes2Net_ASVspoof_ITW.git"
NES2NET_REPO_DIR = Path("/kaggle/working/Nes2Net_ASVspoof_ITW")
XLSR_300M_URL = "https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr2_300m.pt"
XLSR_300M_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/xlsr/xlsr2_300m.pt"),
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/ssl_models/xlsr2_300m.pt"),
    Path("/kaggle/input/sdd-survey/checkpoints/xlsr/xlsr2_300m.pt"),
    Path("/kaggle/input/xlsr-300m/xlsr2_300m.pt"),
]
XLSR_300M_WORKING_CANDIDATES = [
    Path("/kaggle/working/xlsr2_300m.pt"),
]
FAIRSEQ_COMMIT = "a54021305d6b3c4c5959ac9395135f63202db8f1"
FAIRSEQ_REPO_URL = "https://github.com/pytorch/fairseq.git"
FAIRSEQ_SRC_DIR = Path(f"/kaggle/working/fairseq-{FAIRSEQ_COMMIT}")
NES2NET_CKPT_GDRIVE_ID = "1JFGv_2TONMnTLGbiOIuHFfMvuo4SIIpg"
NES2NET_CKPT_WORKING_DIR = Path("/kaggle/working/wav2vec2_nes2net")
NES2NET_CKPT_WORKING_PATH = NES2NET_CKPT_WORKING_DIR / "pretrained_nes2net.pth"
XLSR_AASIST_CKPT_BASES = [
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/xlsr_aasist"),
    Path("/kaggle/input/sdd-survey/checkpoints/xlsr_aasist"),
    Path("/kaggle/input/xlsr-aasist-antispoofing"),
]
XLSR_AASIST_CKPT_NAMES = ["Best_LA_model_for_DF.pth", "Best_LA_model_for_LA.pth", "Best_LA_model_for_ITW.pth"]
NES2NET_CKPT_BASES = [
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/wav2vec2_nes2net"),
    Path("/kaggle/input/sdd-survey/checkpoints/wav2vec2_nes2net"),
    Path("/kaggle/input/wav2vec2-nes2net"),
    Path("/kaggle/input/nes2net-asvspoof-itw"),
    NES2NET_CKPT_WORKING_DIR,
    NES2NET_REPO_DIR,
    Path("/kaggle/working"),
]
LFCC_CKPT_CANDIDATES = [
    Path("/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
    Path("/kaggle/input/sdd-survey/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
    Path("/kaggle/working/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
    Path("results/checkpoints/lfcc_lcnn/lfcc_lcnn.pth"),
]


def _swap_sys_path(repo_dir: Path) -> None:
    repo_str = str(repo_dir)
    if repo_str in sys.path:
        sys.path.remove(repo_str)
    sys.path.insert(0, repo_str)


def ensure_ssl_aasist_repo() -> Path:
    """Clone TakHemlata/SSL_Anti-spoofing only when missing."""
    if not SSL_AASIST_REPO_DIR.exists():
        run_shell(f"git clone {SSL_AASIST_REPO_URL} {SSL_AASIST_REPO_DIR}")
    _swap_sys_path(SSL_AASIST_REPO_DIR)
    return SSL_AASIST_REPO_DIR


def ensure_nes2net_repo() -> Path:
    """Clone Liu-Tianchi/Nes2Net_ASVspoof_ITW only when missing."""
    if not NES2NET_REPO_DIR.exists():
        run_shell(f"git clone {NES2NET_REPO_URL} {NES2NET_REPO_DIR}")
    _swap_sys_path(NES2NET_REPO_DIR)
    return NES2NET_REPO_DIR


def ensure_repo_fairseq(repo_dir: Path) -> None:
    """Install the fairseq snapshot required by the SSL front-ends.

    Nes2Net does not always vendor fairseq, but it was written against the same
    old snapshot as SSL_Anti-spoofing. Installing PyPI fairseq on Kaggle's
    Python 3.12 fails on old omegaconf metadata, so we import a patched pinned
    source tree directly instead of falling back to PyPI.
    """
    fairseq_dir = ensure_fairseq_source(repo_dir)
    patch_fairseq_for_python312(fairseq_dir)
    print(f"installing pinned fairseq from {fairseq_dir}")
    _swap_sys_path(fairseq_dir)

    # pip>=24.1 rejects the omegaconf 2.0.x metadata required by this fairseq
    # snapshot. Keep old pip, but avoid building fairseq itself: the patched
    # source tree is already on sys.path and the SSL loaders only need imports.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pip<24.1"])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "setuptools>=68,<70",
        "wheel",
        "cython",
        "numpy",
    ])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "bitarray",
        "cffi",
        "regex",
        "sacrebleu==1.5.1",
        "tqdm",
        "PyYAML",
        "antlr4-python3-runtime==4.8",
    ])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "omegaconf==2.0.6",
        "hydra-core==1.0.7",
    ])

    patch_fairseq_for_python312(fairseq_dir)
    patch_hydra_for_python312()
    _reset_module_namespace(["fairseq", "hydra", "omegaconf"])
    importlib.invalidate_caches()
    try:
        import fairseq  # noqa: F401
        print(f"fairseq import path: {fairseq.__file__}")
    except Exception as exc:
        raise RuntimeError(
            f"fairseq install finished but import still failed. "
            f"fairseq_dir={fairseq_dir}, sys.path[0]={sys.path[0]}"
        ) from exc


def ensure_fairseq_source(repo_dir: Path) -> Path:
    """Return a local fairseq source tree pinned to the XLS-R-compatible commit."""
    candidates = sorted(
        p for p in repo_dir.iterdir()
        if p.is_dir() and p.name.startswith("fairseq") and (p / "fairseq").exists()
    )
    if candidates:
        return candidates[0]

    if not FAIRSEQ_SRC_DIR.exists():
        print(f"cloning pinned fairseq source: {FAIRSEQ_COMMIT}")
        subprocess.check_call(["git", "clone", "-q", FAIRSEQ_REPO_URL, str(FAIRSEQ_SRC_DIR)])

    # Re-checkout on every run so a reused Kaggle working directory cannot drift.
    subprocess.check_call(["git", "-C", str(FAIRSEQ_SRC_DIR), "checkout", "-q", FAIRSEQ_COMMIT])
    return FAIRSEQ_SRC_DIR


def patch_fairseq_for_python312(fairseq_dir: Path) -> None:
    """Patch old fairseq dataclasses so they import on Python 3.11/3.12.

    The pinned fairseq snapshot uses mutable dataclass defaults such as
    `common: CommonConfig = CommonConfig()`. Newer Python rejects that pattern.
    For this notebook we only rewrite config defaults to `field(default_factory=...)`.
    """
    patch_targets = [
        fairseq_dir / "fairseq" / "dataclass" / "configs.py",
        fairseq_dir / "fairseq" / "models" / "transformer" / "transformer_config.py",
    ]
    for patch_target in patch_targets:
        if not patch_target.exists():
            continue
        n_replacements = patch_dataclass_mutable_defaults(patch_target)
        if n_replacements:
            print(f"patched {n_replacements} fairseq dataclass defaults for Python 3.12: {patch_target}")
    patch_fairseq_hydra_init_for_default_factory(fairseq_dir)
    patch_fairseq_numpy_aliases(fairseq_dir)


def patch_fairseq_hydra_init_for_default_factory(fairseq_dir: Path) -> None:
    """Make fairseq hydra_init understand dataclass default_factory fields.

    After Python 3.12 compatibility patching, fields such as `common` no longer
    have `.default`; they have `.default_factory`. Old fairseq hydra_init only
    reads `.default`, so it passes dataclasses.MISSING into OmegaConf. This patch
    instantiates default_factory values before registering them with Hydra.
    """
    init_path = fairseq_dir / "fairseq" / "dataclass" / "initialize.py"
    if not init_path.exists():
        return
    text = init_path.read_text(encoding="utf-8")
    if "patched_py312_default_factory" in text:
        return

    if "from dataclasses import MISSING" not in text:
        text = "from dataclasses import MISSING\n" + text

    old = "v = FairseqConfig.__dataclass_fields__[k].default"
    new = (
        "field_info = FairseqConfig.__dataclass_fields__[k]\n"
        "        if field_info.default is not MISSING:\n"
        "            v = field_info.default\n"
        "        elif field_info.default_factory is not MISSING:\n"
        "            v = field_info.default_factory()\n"
        "        else:\n"
        "            v = MISSING"
    )
    if old in text:
        text = text.replace(old, new)
        text += "\n# patched_py312_default_factory\n"
        init_path.write_text(text, encoding="utf-8")
        print(f"patched fairseq hydra_init default_factory handling: {init_path}")


def patch_hydra_for_python312() -> None:
    """Patch hydra 1.0.x dataclass defaults installed as a fairseq dependency."""
    spec = importlib.util.find_spec("hydra")
    if spec is None or not spec.submodule_search_locations:
        return
    hydra_root = Path(list(spec.submodule_search_locations)[0])
    conf_path = hydra_root / "conf" / "__init__.py"
    if not conf_path.exists():
        return
    n_replacements = patch_dataclass_mutable_defaults(conf_path)
    if n_replacements:
        print(f"patched {n_replacements} hydra dataclass defaults for Python 3.12: {conf_path}")


def patch_dataclass_mutable_defaults(path: Path) -> int:
    """Rewrite common mutable dataclass defaults for Python 3.11/3.12 compatibility."""
    import re

    text = path.read_text(encoding="utf-8")
    original = text

    n_total = 0
    # Pattern: name: SomeConfig = SomeConfig()
    obj_pattern = re.compile(
        r"^(?P<indent>\s*)(?P<name>\w+):\s+(?P<cls>[\w.]+)\s*=\s*(?P=cls)\(\)\s*$",
        flags=re.MULTILINE,
    )
    text, n_obj = obj_pattern.subn(
        lambda m: (
            f"{m.group('indent')}{m.group('name')}: {m.group('cls')} = "
            f"field(default_factory={m.group('cls')})"
        ),
        text,
    )
    n_total += n_obj

    # Pattern: name: SomeConfig = field(default=SomeConfig())
    field_obj_pattern = re.compile(
        r"^(?P<indent>\s*)(?P<name>\w+):\s+(?P<cls>[\w.]+)\s*=\s*field\(default=(?P=cls)\(\)\)\s*$",
        flags=re.MULTILINE,
    )
    text, n_field_obj = field_obj_pattern.subn(
        lambda m: (
            f"{m.group('indent')}{m.group('name')}: {m.group('cls')} = "
            f"field(default_factory={m.group('cls')})"
        ),
        text,
    )
    n_total += n_field_obj

    # Pattern: name: List[T] = [] / name: Dict[K, V] = {}
    list_pattern = re.compile(r"^(?P<indent>\s*)(?P<name>\w+):\s+List\[(?P<t>[^\]]+)\]\s*=\s*\[\]\s*$", flags=re.MULTILINE)
    text, n_list = list_pattern.subn(
        lambda m: f"{m.group('indent')}{m.group('name')}: List[{m.group('t')}] = field(default_factory=list)",
        text,
    )
    n_total += n_list

    dict_pattern = re.compile(r"^(?P<indent>\s*)(?P<name>\w+):\s+Dict\[(?P<t>[^\]]+)\]\s*=\s*\{\}\s*$", flags=re.MULTILINE)
    text, n_dict = dict_pattern.subn(
        lambda m: f"{m.group('indent')}{m.group('name')}: Dict[{m.group('t')}] = field(default_factory=dict)",
        text,
    )
    n_total += n_dict

    if n_total:
        # Add the import only when a rewrite actually needs field(...). This
        # avoids touching unrelated files and keeps __future__ imports valid.
        dataclasses_import = text.split("from dataclasses import", 1)
        if len(dataclasses_import) == 2:
            first_line = dataclasses_import[1].split("\n", 1)[0]
            if "field" not in first_line:
                text = re.sub(
                    r"^(from dataclasses import )([^\n]+)$",
                    lambda m: m.group(1) + m.group(2).rstrip() + ", field",
                    text,
                    count=1,
                    flags=re.MULTILINE,
                )
        else:
            future_imports = list(re.finditer(r"^from __future__ import [^\n]+\n", text, flags=re.MULTILINE))
            if future_imports:
                insert_at = future_imports[-1].end()
                text = text[:insert_at] + "from dataclasses import field\n" + text[insert_at:]
            else:
                text = "from dataclasses import field\n" + text

    if text != original:
        path.write_text(text, encoding="utf-8")
    return n_total


def patch_fairseq_numpy_aliases(fairseq_dir: Path) -> None:
    """Replace removed numpy scalar type aliases across the fairseq bundle.

    NumPy 1.24 removed np.float, np.int, np.bool, np.complex, np.object, np.str.
    The pinned fairseq snapshot still uses these in several files (e.g. indexed_dataset.py).
    """
    import re
    _aliases = [
        (re.compile(r'\bnp\.float\b'), 'np.float64'),
        (re.compile(r'\bnp\.int\b'), 'np.int_'),
        (re.compile(r'\bnp\.bool\b'), 'np.bool_'),
        (re.compile(r'\bnp\.complex\b'), 'np.complex128'),
        (re.compile(r'\bnp\.object\b'), 'object'),
        (re.compile(r'\bnp\.str\b'), 'np.str_'),
    ]
    total = 0
    for py_file in fairseq_dir.rglob("*.py"):
        try:
            text = py_file.read_text(encoding="utf-8")
        except Exception:
            continue
        original = text
        for pattern, replacement in _aliases:
            text = pattern.sub(replacement, text)
        if text != original:
            py_file.write_text(text, encoding="utf-8")
            total += 1
    if total:
        print(f"patched numpy deprecated aliases in {total} fairseq files")


def find_or_download_xlsr_300m() -> Path:
    """Locate or download the fairseq XLS-R 300M base SSL checkpoint."""
    for path in XLSR_300M_INPUT_CANDIDATES + XLSR_300M_WORKING_CANDIDATES:
        if path.exists():
            print(f"found XLS-R 300M base SSL: {path}")
            return path
    target = Path("/kaggle/working/xlsr2_300m.pt")
    print(f"downloading XLS-R 300M (~1.2 GB) to {target}")
    run_shell(f"wget -q -O {target} {XLSR_300M_URL}")
    return target


def _link_xlsr_into(repo_dir: Path) -> None:
    """Both SSL_Anti-spoofing and Nes2Net hard-code ./xlsr2_300m.pt — point it at the cached file."""
    target = repo_dir / "xlsr2_300m.pt"
    if target.exists():
        return
    src = find_or_download_xlsr_300m()
    try:
        target.symlink_to(src)
    except (OSError, NotImplementedError):
        import shutil
        shutil.copy(src, target)


def find_xlsr_aasist_checkpoint() -> Path:
    """Locate TakHemlata's pretrained Wav2Vec2-XLSR+AASIST anti-spoofing weights."""
    for base in XLSR_AASIST_CKPT_BASES:
        for name in XLSR_AASIST_CKPT_NAMES:
            if (base / name).exists():
                return base / name
        if base.exists():
            for path in base.glob("*.pth"):
                return path
    raise FileNotFoundError(
        "Wav2Vec2-XLSR+AASIST pretrained weights not found. "
        "Upload TakHemlata 'Best_LA_model_for_DF.pth' as a Kaggle dataset under "
        "/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/xlsr_aasist/."
    )


def find_xlsr_nes2net_checkpoint() -> Path:
    """Locate Liu-Tianchi's pretrained wav2vec2+Nes2Net-X anti-spoofing weights."""
    def is_nes2net_checkpoint(path: Path) -> bool:
        name = path.name.lower()
        if name == "xlsr2_300m.pt":
            return False
        return (
            name == "pretrained_nes2net.pth"
            or "avg_ckpt" in name
            or ("nes2net" in name and path.suffix.lower() in {".pth", ".pt"})
        )

    for base in NES2NET_CKPT_BASES:
        if base.exists():
            for pattern in ("*avg_ckpt*.pth", "pretrained_nes2net.pth", "*Nes2Net*.pth", "*nes2net*.pth"):
                for path in sorted(base.glob(pattern)):
                    if is_nes2net_checkpoint(path):
                        return path

    # Prefer immutable Kaggle input datasets above. If the checkpoint was not
    # attached, download the official Google Drive file into /kaggle/working.
    # This matches the manual Colab/Kaggle command previously used for Nes2Net.
    NES2NET_CKPT_WORKING_DIR.mkdir(parents=True, exist_ok=True)
    try:
        import gdown  # type: ignore
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown  # type: ignore

    print(f"downloading wav2vec2+Nes2Net checkpoint to {NES2NET_CKPT_WORKING_PATH}")
    gdown.download(
        id=NES2NET_CKPT_GDRIVE_ID,
        output=str(NES2NET_CKPT_WORKING_PATH),
        quiet=False,
    )
    if NES2NET_CKPT_WORKING_PATH.exists() and NES2NET_CKPT_WORKING_PATH.stat().st_size > 0:
        return NES2NET_CKPT_WORKING_PATH

    raise FileNotFoundError(
        "wav2vec2+Nes2Net pretrained weights not found. "
        "Attach/upload Liu-Tianchi's averaged checkpoint as a Kaggle dataset under "
        "/kaggle/input/datasets/minhbhm/sdd-survey/checkpoints/wav2vec2_nes2net/, "
        f"or enable Internet so gdown can download Google Drive id {NES2NET_CKPT_GDRIVE_ID}."
    )


def _strip_state_prefix(state: dict) -> dict:
    out = {}
    for k, v in state.items():
        if k.startswith("module."):
            out[k[len("module."):]] = v
        else:
            out[k] = v
    return out


def _reset_module_namespace(prefixes: list[str]) -> None:
    for name in list(sys.modules):
        if name in prefixes or any(name.startswith(p + ".") for p in prefixes):
            del sys.modules[name]


def _load_required_state(model: nn.Module, state: dict, model_name: str) -> None:
    """Load a checkpoint and fail on any architecture mismatch.

    Survey EER values are only useful when the full pretrained checkpoint is
    loaded. Silent partial loading can look successful while leaving random
    layers in the model, so missing/unexpected keys are fatal.
    """
    missing, unexpected = model.load_state_dict(_strip_state_prefix(state), strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"{model_name} checkpoint does not match the model architecture. "
            f"missing={list(missing)[:10]} unexpected={list(unexpected)[:10]}"
        )


def load_xlsr_aasist_model() -> nn.Module:
    """Load Wav2Vec2-XLSR (300M) + AASIST from TakHemlata/SSL_Anti-spoofing."""
    repo_dir = ensure_ssl_aasist_repo()
    ensure_repo_fairseq(repo_dir)
    _link_xlsr_into(repo_dir)
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["model", "models"])
        from model import Model as XLSRAASIST  # type: ignore
        import argparse
        model = XLSRAASIST(argparse.Namespace(), DEVICE).to(DEVICE)
        ckpt = find_xlsr_aasist_checkpoint()
        print(f"loading Wav2Vec2-XLSR+AASIST weights: {ckpt}")
        state = torch.load(ckpt, map_location=DEVICE)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        _load_required_state(model, state, "XLS-R+AASIST")
        return model.eval()
    finally:
        os.chdir(cwd)


def load_xlsr_nes2net_model() -> nn.Module:
    """Load wav2vec2 (XLS-R 300M) + Nes2Net-X from Liu-Tianchi/Nes2Net_ASVspoof_ITW."""
    repo_dir = ensure_nes2net_repo()
    ensure_repo_fairseq(repo_dir)
    _link_xlsr_into(repo_dir)
    cwd = Path.cwd()
    try:
        os.chdir(repo_dir)
        _reset_module_namespace(["model_scripts"])
        from model_scripts.wav2vec2_Nes2Net_X import wav2vec2_Nes2Net_no_Res_w_allT  # type: ignore
        import argparse
        # Defaults from the repo CLI. SE_ratio uses nargs="+", so keep it as a
        # one-item list; the model indexes SE_ratio[0] during construction.
        # --pool_func mean --SE_ratio 1 --Nes_ratio 8 8
        args = argparse.Namespace(
            n_output_logits=2,
            Nes_ratio=[8, 8],
            dilation=2,
            pool_func="mean",
            SE_ratio=[1],
        )
        model = wav2vec2_Nes2Net_no_Res_w_allT(args, DEVICE).to(DEVICE)
        ckpt = find_xlsr_nes2net_checkpoint()
        print(f"loading wav2vec2+Nes2Net-X weights: {ckpt}")
        state = torch.load(ckpt, map_location=DEVICE)
        if isinstance(state, dict) and "model_state_dict" in state:
            state = state["model_state_dict"]
        elif isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        _load_required_state(model, state, "XLS-R+Nes2Net")
        return model.eval()
    finally:
        os.chdir(cwd)


@torch.no_grad()
def predict_ssl_pair_batch(model, waves: torch.Tensor) -> np.ndarray:
    """SSL models return 2-class logits ordered [spoof, bonafide]."""
    logits = model(waves.to(DEVICE))
    if not torch.is_tensor(logits) or logits.dim() != 2 or logits.shape[1] != 2:
        raise RuntimeError(f"Unexpected SSL model output shape/type: {type(logits)} {getattr(logits, 'shape', None)}")
    return torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()


def find_lfcc_checkpoint() -> Path:
    for path in LFCC_CKPT_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError("LFCC+LCNN checkpoint was not found. Attach sdd-survey or copy lfcc_lcnn.pth.")


def _first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def _find_xlsr_aasist_checkpoint_no_download() -> Path | None:
    for base in XLSR_AASIST_CKPT_BASES:
        for name in XLSR_AASIST_CKPT_NAMES:
            candidate = base / name
            if candidate.exists():
                return candidate
        if base.exists():
            for candidate in base.glob("*.pth"):
                return candidate
    return None


def _find_nes2net_checkpoint_no_download() -> Path | None:
    for base in NES2NET_CKPT_BASES:
        if not base.exists():
            continue
        for pattern in ("*avg_ckpt*.pth", "pretrained_nes2net.pth", "*Nes2Net*.pth", "*nes2net*.pth"):
            for candidate in sorted(base.glob(pattern)):
                name = candidate.name.lower()
                if name != "xlsr2_300m.pt":
                    return candidate
    return None


def preflight_check_model_inputs(
    enabled_models: list[str],
    results: dict | None = None,
    force_eval: dict[str, bool] | None = None,
    require_kaggle_xlsr: bool = True,
) -> None:
    """Print model asset availability and fail early for missing required files."""
    results = results or {}
    force_eval = force_eval or {}
    missing: list[str] = []
    xlsr_input = _first_existing(XLSR_300M_INPUT_CANDIDATES)
    xlsr_any = xlsr_input or _first_existing(XLSR_300M_WORKING_CANDIDATES)

    print("model input preflight")
    for model_name in enabled_models:
        if model_name in results and not force_eval.get(model_name, False):
            print(f"  OK   {model_name}: cached in results.pkl")
            continue

        if model_name in {"AASIST", "AASIST-L"}:
            print(f"  INFO {model_name}: official weights are loaded from the cloned clovaai/aasist repo")
        elif model_name == "AASIST3":
            print("  INFO AASIST3: checkpoint is loaded through Hugging Face model MTUCI/AASIST3")
        elif model_name == "LFCC+LCNN":
            ckpt = _first_existing(LFCC_CKPT_CANDIDATES)
            if ckpt:
                print(f"  OK   LFCC+LCNN checkpoint: {ckpt}")
            else:
                missing.append("LFCC+LCNN checkpoint lfcc_lcnn.pth")
        elif model_name == "XLS-R+AASIST":
            ckpt = _find_xlsr_aasist_checkpoint_no_download()
            if xlsr_input:
                print(f"  OK   XLS-R base checkpoint: {xlsr_input}")
            elif xlsr_any and not require_kaggle_xlsr:
                print(f"  OK   XLS-R base checkpoint: {xlsr_any}")
            else:
                missing.append("XLS-R base checkpoint xlsr2_300m.pt for XLS-R+AASIST")
            if ckpt:
                print(f"  OK   XLS-R+AASIST checkpoint: {ckpt}")
            else:
                missing.append("XLS-R+AASIST checkpoint Best_LA_model_for_*.pth")
        elif model_name == "XLS-R+Nes2Net":
            ckpt = _find_nes2net_checkpoint_no_download()
            if xlsr_input:
                print(f"  OK   XLS-R base checkpoint: {xlsr_input}")
            elif xlsr_any and not require_kaggle_xlsr:
                print(f"  OK   XLS-R base checkpoint: {xlsr_any}")
            else:
                missing.append("XLS-R base checkpoint xlsr2_300m.pt for XLS-R+Nes2Net")
            if ckpt:
                print(f"  OK   XLS-R+Nes2Net checkpoint: {ckpt}")
            else:
                missing.append("XLS-R+Nes2Net checkpoint pretrained_nes2net.pth")
        else:
            print(f"  WARN {model_name}: no preflight rule")

    if missing:
        message = "\n".join(f"  - {item}" for item in sorted(set(missing)))
        raise FileNotFoundError(
            "Missing model inputs before evaluation:\n"
            f"{message}\n"
            "Attach the missing files as Kaggle input datasets before running this notebook. "
            f"XLS-R 300M URL: {XLSR_300M_URL}"
        )


def load_lfcc_lcnn_model() -> nn.Module:
    model = SimpleLCNN().to(DEVICE)
    ckpt = find_lfcc_checkpoint()
    print(f"loading LFCC+LCNN checkpoint: {ckpt}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model.eval()


def load_audio_16k(audio_ref):
    """Load path, bytes, or HF audio dict into mono 16 kHz waveform with shape (1, T)."""
    if isinstance(audio_ref, (str, Path)):
        wav, sr = torchaudio.load(str(audio_ref))
    elif isinstance(audio_ref, bytes):
        array, sr = sf.read(io.BytesIO(audio_ref), dtype="float32")
        if array.ndim == 1:
            wav = torch.from_numpy(array).float().unsqueeze(0)
        else:
            wav = torch.from_numpy(array).float().T
    elif isinstance(audio_ref, dict) and "array" in audio_ref and "sampling_rate" in audio_ref:
        wav = torch.from_numpy(audio_ref["array"]).float().unsqueeze(0)
        sr = int(audio_ref["sampling_rate"])
    else:
        raise TypeError(f"unsupported audio ref: {type(audio_ref)}")

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16000:
        wav = torchaudio.transforms.Resample(sr, 16000)(wav)
    return wav


def prepare_waveform(wav: torch.Tensor, cut: int = 64600) -> torch.Tensor:
    if wav.shape[1] < cut:
        wav = F.pad(wav, (0, cut - wav.shape[1]))
    else:
        wav = wav[:, :cut]
    return wav.squeeze(0)


def prepare_lfcc(wav: torch.Tensor, max_frames: int = 400) -> torch.Tensor:
    feat = LFCC_TRANSFORM(wav)
    if feat.shape[2] < max_frames:
        feat = F.pad(feat, (0, max_frames - feat.shape[2]))
    else:
        feat = feat[:, :, :max_frames]
    return feat.squeeze(0)


class WaveformDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        try:
            wav = prepare_waveform(load_audio_16k(row["audio_path"]))
            return wav, int(row["label"]), str(row["utt_id"]), 0
        except Exception:
            return torch.zeros(64600), int(row["label"]), str(row["utt_id"]), 1


class LFCCDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        try:
            feat = prepare_lfcc(load_audio_16k(row["audio_path"]))
            return feat, int(row["label"]), str(row["utt_id"]), 0
        except Exception:
            return torch.zeros(60, 400), int(row["label"]), str(row["utt_id"]), 1


@torch.no_grad()
def predict_aasist_batch(model, waves: torch.Tensor) -> np.ndarray:
    """AASIST and AASIST-L output logits [spoof, bonafide]."""
    _, logits = model(waves.to(DEVICE))
    return logits.softmax(dim=1)[:, 1].detach().cpu().numpy()


@torch.no_grad()
def predict_aasist3_batch(model, waves: torch.Tensor) -> np.ndarray:
    """AASIST3 output logits [spoof, bonafide]. Use index 1 as bonafide score."""
    logits = model(waves.to(DEVICE))
    return torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()


@torch.no_grad()
def predict_lfcc_batch(model, feats: torch.Tensor) -> np.ndarray:
    if feats.dim() == 3:
        feats = feats.unsqueeze(1)
    logits = model(feats.to(DEVICE))
    return torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()


MODEL_REGISTRY = {
    "AASIST": {
        "loader": lambda: load_aasist_model("AASIST.conf", "AASIST.pth"),
        "dataset": WaveformDataset,
        "predict": predict_aasist_batch,
        "batch_size": 64,
    },
    "AASIST-L": {
        "loader": lambda: load_aasist_model("AASIST-L.conf", "AASIST-L.pth"),
        "dataset": WaveformDataset,
        "predict": predict_aasist_batch,
        "batch_size": 64,
    },
    "AASIST3": {
        "loader": load_aasist3_model,
        "dataset": WaveformDataset,
        "predict": predict_aasist3_batch,
        "batch_size": 64,
    },
    "LFCC+LCNN": {
        "loader": load_lfcc_lcnn_model,
        "dataset": LFCCDataset,
        "predict": predict_lfcc_batch,
        "batch_size": 128,
    },
    "XLS-R+AASIST": {
        "loader": load_xlsr_aasist_model,
        "dataset": WaveformDataset,
        "predict": predict_ssl_pair_batch,
        "batch_size": 8,
    },
    "XLS-R+Nes2Net": {
        "loader": load_xlsr_nes2net_model,
        "dataset": WaveformDataset,
        "predict": predict_ssl_pair_batch,
        "batch_size": 8,
    },
}


COMPOUND_GROUPS = [
    ("WaveformDataset", ["XLS-R+AASIST", "XLS-R+Nes2Net"]),
]

In [ ]:
def evaluate_model_on_dataframe(
    model_name: str,
    df: pd.DataFrame,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: bool = False,
    partial_save_every: int = 5000,
    num_workers: int = 2,
    partial_input_dirs: list[Path] | None = None,
) -> dict:
    """Evaluate one model on local files with partial resume and unified pkl writes."""
    if model_name in results and not force_eval:
        print(f"{model_name}: already present in results.pkl, skipping")
        return results[model_name]

    entry = MODEL_REGISTRY[model_name]
    partial = load_partial(output_dir, model_name, partial_input_dirs)
    scores, labels, utt_ids = [], [], []
    completed = set()
    if partial:
        scores = partial["scores"]
        labels = partial["labels"]
        utt_ids = partial["utt_ids"]
        completed = set(utt_ids)
        print(f"{model_name}: resume partial with {len(completed):,} completed utterances")

    todo_df = df[~df["utt_id"].astype(str).isin(completed)].reset_index(drop=True)
    print(f"{model_name}: todo={len(todo_df):,} already_done={len(completed):,}")

    model = entry["loader"]()
    dataset = entry["dataset"](todo_df)
    loader = DataLoader(
        dataset,
        batch_size=entry["batch_size"],
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    errors = 0
    last_save = len(utt_ids)
    t0 = time.time()
    try:
        for batch_inputs, batch_labels, batch_utt_ids, batch_is_error in tqdm(loader, desc=model_name):
            batch_scores = entry["predict"](model, batch_inputs)
            ok_mask = batch_is_error.numpy() == 0
            errors += int((~ok_mask).sum())
            for i, ok in enumerate(ok_mask):
                if not ok:
                    continue
                scores.append(float(batch_scores[i]))
                labels.append(int(batch_labels[i].item()))
                utt_ids.append(str(batch_utt_ids[i]))

            if len(utt_ids) - last_save >= partial_save_every:
                save_partial(output_dir, model_name, scores, labels, utt_ids)
                last_save = len(utt_ids)
    except KeyboardInterrupt:
        save_partial(output_dir, model_name, scores, labels, utt_ids)
        print(f"{model_name}: interrupted; partial saved")
        raise
    finally:
        release_model(model)
        model = None

    eer = compute_eer(scores, labels)
    result = {
        "eer": eer,
        "scores": np.asarray(scores, dtype=np.float64),
        "labels": np.asarray(labels, dtype=np.int64),
    }
    results[model_name] = result
    save_results_pickle(results, output_pkl)
    clear_partial(output_dir, model_name)
    print(f"{model_name}: EER={eer:.4f}% N={len(scores):,} errors={errors:,} elapsed={(time.time()-t0)/60:.1f} min")
    return result


def evaluate_models_compound(
    model_names: list[str],
    df: pd.DataFrame,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: dict[str, bool] | None = None,
    partial_save_every: int = 5000,
    num_workers: int = 2,
    partial_input_dirs: list[Path] | None = None,
) -> None:
    """Run multiple models in one pass over the same data loader.

    All listed models must share the same dataset class. Each model has its own
    partial.npz checkpoint and is added to results.pkl independently when finished.
    The compound runner saves audio I/O versus running models sequentially because
    each utterance is decoded once and each model in the group consumes the same
    batch tensor. The XLS-R front-ends inside SSL models are NOT shared because
    each pretrained checkpoint contains independently fine-tuned SSL weights.
    """
    force_eval = force_eval or {}
    pending = []
    for name in model_names:
        if name in results and not force_eval.get(name, False):
            print(f"{name}: already present in results.pkl, skipping")
        else:
            pending.append(name)
    if not pending:
        return

    dataset_classes = {MODEL_REGISTRY[n]["dataset"] for n in pending}
    if len(dataset_classes) > 1:
        raise ValueError(f"compound run requires identical dataset class, got {dataset_classes}")
    DatasetCls = dataset_classes.pop()

    partials = {}
    completed = {}
    for name in pending:
        partial = load_partial(output_dir, name, partial_input_dirs)
        if partial:
            partials[name] = partial
            completed[name] = set(partial["utt_ids"])
            print(f"{name}: resume partial with {len(partial['utt_ids']):,} completed")
        else:
            partials[name] = {"scores": [], "labels": [], "utt_ids": []}
            completed[name] = set()

    todo_mask = df["utt_id"].astype(str).map(
        lambda u: any(u not in completed[n] for n in pending)
    )
    todo_df = df[todo_mask].reset_index(drop=True)
    print(f"compound[{'+'.join(pending)}]: todo={len(todo_df):,}")

    models = {}
    for name in pending:
        print(f"loading {name}")
        models[name] = MODEL_REGISTRY[name]["loader"]()
    batch_size = min(MODEL_REGISTRY[n]["batch_size"] for n in pending)

    dataset = DatasetCls(todo_df)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    last_save = {n: len(partials[n]["utt_ids"]) for n in pending}
    errors = 0
    t0 = time.time()
    desc = "+".join(pending)
    try:
        for batch_inputs, batch_labels, batch_utt_ids, batch_is_error in tqdm(loader, desc=desc):
            ok_mask = batch_is_error.numpy() == 0
            errors += int((~ok_mask).sum())
            for name in pending:
                entry = MODEL_REGISTRY[name]
                batch_scores = entry["predict"](models[name], batch_inputs)
                for i, ok in enumerate(ok_mask):
                    if not ok:
                        continue
                    utt_id = str(batch_utt_ids[i])
                    if utt_id in completed[name]:
                        continue
                    partials[name]["scores"].append(float(batch_scores[i]))
                    partials[name]["labels"].append(int(batch_labels[i].item()))
                    partials[name]["utt_ids"].append(utt_id)
                    completed[name].add(utt_id)
                if len(partials[name]["utt_ids"]) - last_save[name] >= partial_save_every:
                    save_partial(output_dir, name, partials[name]["scores"], partials[name]["labels"], partials[name]["utt_ids"])
                    last_save[name] = len(partials[name]["utt_ids"])
    except KeyboardInterrupt:
        for name in pending:
            save_partial(output_dir, name, partials[name]["scores"], partials[name]["labels"], partials[name]["utt_ids"])
        print("compound: interrupted; partials saved")
        raise
    finally:
        for name in pending:
            release_model(models[name])
            models[name] = None

    for name in pending:
        eer = compute_eer(partials[name]["scores"], partials[name]["labels"])
        results[name] = {
            "eer": eer,
            "scores": np.asarray(partials[name]["scores"], dtype=np.float64),
            "labels": np.asarray(partials[name]["labels"], dtype=np.int64),
        }
        save_results_pickle(results, output_pkl)
        clear_partial(output_dir, name)
        print(f"{name}: EER={eer:.4f}% N={len(partials[name]['scores']):,}")
    print(f"compound[{desc}] elapsed={(time.time()-t0)/60:.1f} min, errors={errors:,}")


def run_eval_plan(
    enabled_models: list[str],
    df: pd.DataFrame,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: dict[str, bool] | None = None,
    partial_save_every: int = 5000,
    num_workers: int = 2,
    compound_enabled: bool = False,
    partial_input_dirs: list[Path] | None = None,
) -> None:
    """Schedule single-model and compound runs for the configured ENABLED_MODELS.

    Compound mode is opt-in because loading two XLS-R models at once can exceed Kaggle
    T4/P100 memory. When disabled, every model runs sequentially with the same partial
    resume behavior. When enabled, models declared in COMPOUND_GROUPS share one loader.
    """
    force_eval = force_eval or {}
    grouped = set()
    if compound_enabled:
        for _tag, group in COMPOUND_GROUPS:
            members = [m for m in group if m in enabled_models]
            if len(members) >= 2:
                evaluate_models_compound(
                    model_names=members,
                    df=df,
                    results=results,
                    output_pkl=output_pkl,
                    output_dir=output_dir,
                    force_eval=force_eval,
                    partial_save_every=partial_save_every,
                    num_workers=num_workers,
                    partial_input_dirs=partial_input_dirs,
                )
                grouped.update(members)
    for name in enabled_models:
        if name in grouped:
            continue
        evaluate_model_on_dataframe(
            model_name=name,
            df=df,
            results=results,
            output_pkl=output_pkl,
            output_dir=output_dir,
            force_eval=force_eval.get(name, False),
            partial_save_every=partial_save_every,
            num_workers=num_workers,
            partial_input_dirs=partial_input_dirs,
        )

In [ ]:
ASV5_HF_REPO_ID = "jungjee/asvspoof5"
ASV5_CACHE_DIR = Path("/kaggle/working/asvspoof5_hf_cache")
ASV5_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def ensure_hf_hub():
    """Import huggingface_hub, installing it when Kaggle image does not include it."""
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import hf_hub_download
    return hf_hub_download


def hf_download_asv5_file(filename: str) -> Path:
    """Download one ASVspoof 5 file from the HF repo cache."""
    hf_hub_download = ensure_hf_hub()
    path = hf_hub_download(
        repo_id=ASV5_HF_REPO_ID,
        repo_type="dataset",
        filename=filename,
        cache_dir=str(ASV5_CACHE_DIR),
    )
    return Path(path)


def hf_tar_names_for_split(split: str) -> list[str]:
    """Return ASVspoof 5 FLAC tar filenames for one split."""
    if split == "train":
        return [f"flac_T_{suffix}.tar" for suffix in ["aa", "ab", "ac", "ad", "ae"]]
    if split == "dev":
        return [f"flac_D_{suffix}.tar" for suffix in ["aa", "ab", "ac"]]
    if split == "eval":
        return [f"flac_E_{suffix}.tar" for suffix in ["aa", "ab", "ac", "ad", "ae", "af", "ag", "ah", "ai", "aj"]]
    raise ValueError(f"Unknown ASVspoof 5 split: {split}")


def protocol_member_candidates(split: str) -> list[str]:
    if split == "train":
        return ["ASVspoof5.train.tsv", "train.tsv"]
    return [f"ASVspoof5.{split}.track_1.tsv", f"{split}.track_1.tsv"]


def read_protocol_from_hf_tar(split: str) -> str:
    """Read Track 1 protocol text from ASVspoof5_protocols.tar."""
    proto_tar = hf_download_asv5_file("ASVspoof5_protocols.tar")
    wanted = protocol_member_candidates(split)
    with tarfile.open(proto_tar, "r:*") as tar:
        names = tar.getnames()
        for member in tar:
            base = Path(member.name).name
            if base in wanted:
                extracted = tar.extractfile(member)
                if extracted is None:
                    continue
                return extracted.read().decode("utf-8")
    raise FileNotFoundError(f"Could not find {wanted} in {proto_tar}. Members include: {names[:20]}")


def load_asv5_hf_tar_protocol(split: str) -> dict:
    """Load Track 1 protocol rows directly from the protocol tar file."""
    text = read_protocol_from_hf_tar(split)
    meta = {}
    for line in text.splitlines():
        parts = line.strip().split()
        if len(parts) < 10:
            continue
        speaker, utt_id, gender, codec, codec_q, codec_seed, attack_tag, attack_label, key, tmp = parts[:10]
        if key not in ("bonafide", "spoof"):
            continue
        meta[utt_id] = {
            "speaker": speaker,
            "utt_id": utt_id,
            "gender": gender,
            "codec": "nocodec" if codec == "-" else codec,
            "codec_q": codec_q,
            "codec_seed": codec_seed,
            "attack_tag": "bonafide" if attack_tag == "-" else attack_tag,
            "attack": "bonafide" if attack_label == "bonafide" else attack_label,
            "label": 1 if key == "bonafide" else 0,
        }
    if not meta:
        raise RuntimeError(f"Parsed zero ASVspoof 5 protocol rows for split={split}")
    print(f"HF tar protocol rows for {split} Track 1: {len(meta):,}")
    return meta


def iter_asv5_hf_tar_audio(protocol: dict):
    """Yield (utt_id, flac_bytes) from downloaded ASVspoof 5 tar shards."""
    for filename in hf_tar_names_for_split(ASV5_SPLIT):
        tar_path = hf_download_asv5_file(filename)
        print(f"streaming {filename}: {tar_path}")
        with tarfile.open(tar_path, "r:*") as tar:
            for member in tar:
                if not member.isfile():
                    continue
                utt_id = Path(member.name).name
                if utt_id.endswith(".flac"):
                    utt_id = utt_id[:-5]
                if utt_id not in protocol:
                    continue
                extracted = tar.extractfile(member)
                if extracted is None:
                    continue
                yield utt_id, extracted.read()


def build_asv5_hf_tar_index(protocol: dict) -> pd.DataFrame:
    """Scan ASVspoof 5 HF tar shards and verify audio/protocol matching."""
    rows = []
    seen = set()
    for utt_id, _ in tqdm(iter_asv5_hf_tar_audio(protocol), desc="scan_asv5_hf_tar"):
        if utt_id in seen:
            continue
        seen.add(utt_id)
        meta = protocol[utt_id]
        rows.append({
            "utt_id": utt_id,
            "label": int(meta["label"]),
            "speaker": meta.get("speaker", "unknown"),
            "codec": meta.get("codec", "unknown"),
            "attack_tag": meta.get("attack_tag", "unknown"),
            "attack": meta.get("attack", "unknown"),
        })
        if SMOKE_TEST_N is not None and len(rows) >= SMOKE_TEST_N:
            break
        if SMOKE_TEST_N is None and len(rows) >= len(protocol):
            break
    df = pd.DataFrame(rows)
    print("ASVspoof 5 HF tar scan summary")
    print(f"  protocol rows      : {len(protocol):,}")
    print(f"  matched audio rows : {len(df):,}")
    if df.empty:
        raise RuntimeError("ASVspoof 5 HF tar scan matched zero audio rows.")
    print(df.head())
    return df


def predict_tar_batch(model, audio_items: list[tuple[str, bytes, int]], entry: dict):
    """Decode a list of tar FLAC byte payloads and run one model batch."""
    inputs, labels, utt_ids = [], [], []
    for utt_id, audio_bytes, label in audio_items:
        wav = load_audio_16k(audio_bytes)
        if entry["dataset"] is WaveformDataset:
            inputs.append(prepare_waveform(wav))
        else:
            inputs.append(prepare_lfcc(wav))
        labels.append(label)
        utt_ids.append(utt_id)
    batch = torch.stack(inputs)
    return entry["predict"](model, batch), labels, utt_ids


def evaluate_asv5_hf_tar_model(
    model_name: str,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: bool = False,
    partial_save_every: int = 5000,
    partial_input_dirs: list[Path] | None = None,
):
    """Evaluate ASVspoof 5 by reading FLAC bytes directly from HF tar shards."""
    if model_name in results and not force_eval:
        print(f"{model_name}: already present in results.pkl, skipping")
        return results[model_name]

    protocol = ASV5_HF_TAR_PROTOCOL
    target_ids = set(ASV5_HF_TAR_INDEX["utt_id"].astype(str))
    entry = MODEL_REGISTRY[model_name]
    model = entry["loader"]()

    partial = load_partial(output_dir, model_name, partial_input_dirs)
    scores, labels, utt_ids = [], [], []
    completed = set()
    if partial:
        scores, labels, utt_ids = partial["scores"], partial["labels"], partial["utt_ids"]
        completed = set(utt_ids)
        print(f"{model_name}: resume partial with {len(completed):,} completed utterances")

    pending = []
    last_save = len(utt_ids)
    t0 = time.time()
    try:
        for utt_id, audio_bytes in tqdm(iter_asv5_hf_tar_audio(protocol), desc=f"{model_name}:hf_tar"):
            if utt_id not in target_ids or utt_id in completed:
                continue
            pending.append((utt_id, audio_bytes, int(protocol[utt_id]["label"])))
            if len(pending) >= entry["batch_size"]:
                batch_scores, batch_labels, batch_ids = predict_tar_batch(model, pending, entry)
                scores.extend(float(x) for x in batch_scores)
                labels.extend(int(x) for x in batch_labels)
                utt_ids.extend(str(x) for x in batch_ids)
                completed.update(batch_ids)
                pending = []

            if len(utt_ids) - last_save >= partial_save_every:
                save_partial(output_dir, model_name, scores, labels, utt_ids)
                last_save = len(utt_ids)
            if SMOKE_TEST_N is not None and len(utt_ids) >= SMOKE_TEST_N:
                break

        if pending:
            batch_scores, batch_labels, batch_ids = predict_tar_batch(model, pending, entry)
            scores.extend(float(x) for x in batch_scores)
            labels.extend(int(x) for x in batch_labels)
            utt_ids.extend(str(x) for x in batch_ids)
    except KeyboardInterrupt:
        save_partial(output_dir, model_name, scores, labels, utt_ids)
        print(f"{model_name}: interrupted; partial saved")
        raise
    finally:
        release_model(model)
        model = None

    if not scores:
        raise RuntimeError("No ASVspoof 5 HF tar scores were produced.")
    eer = compute_eer(scores, labels)
    result = {"eer": eer, "scores": np.asarray(scores, dtype=np.float64), "labels": np.asarray(labels, dtype=np.int64)}
    results[model_name] = result
    save_results_pickle(results, output_pkl)
    clear_partial(output_dir, model_name)
    print(f"{model_name}: EER={eer:.4f}% N={len(scores):,} elapsed={(time.time()-t0)/60:.1f} min")
    return result


def _decode_protocol_bytes(value) -> str:
    def maybe_b64_decode(text: str) -> str | None:
        padded = text + "=" * (-len(text) % 4)
        try:
            decoded = base64.b64decode(padded).decode("utf-8")
            if "\n" in decoded and " " in decoded:
                return decoded
        except Exception:
            return None
        return None

    if isinstance(value, bytes):
        text = value.decode("utf-8")
        if "\n" in text and " " in text:
            return text
        return maybe_b64_decode(text.strip()) or text
    if isinstance(value, str):
        text = value.strip()
        if "\n" in text and " " in text:
            return text
        # Hugging Face's viewer may expose binary protocol payloads as base64-like
        # strings. Decode defensively when the direct string is not parseable text.
        return maybe_b64_decode(text) or text
    raise TypeError(f"unsupported protocol payload type: {type(value)}")


def load_asv5_hf_protocol(split: str) -> dict:
    """Load Track 1 labels from the protocol sample in the HF WebDataset stream."""
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import load_dataset

    stream = load_dataset("jungjee/asvspoof5", split="train", streaming=True)
    first = next(iter(stream))
    protocol_key = "train.tsv" if split == "train" else f"{split}.track_1.tsv"
    if protocol_key not in first:
        alt = f"ASVspoof5.{split}.track_1.tsv"
        protocol_key = alt if alt in first else protocol_key
    if protocol_key not in first and split == "train" and "ASVspoof5.train.tsv" in first:
        protocol_key = "ASVspoof5.train.tsv"
    if protocol_key not in first:
        raise KeyError(f"Could not find {protocol_key} in ASVspoof 5 HF protocol sample. Keys: {list(first)}")

    meta = {}
    for line in _decode_protocol_bytes(first[protocol_key]).splitlines():
        parts = line.strip().split()
        if len(parts) < 10:
            continue
        speaker, utt_id, gender, codec, codec_q, codec_seed, attack_tag, attack_label, key, tmp = parts[:10]
        if key not in ("bonafide", "spoof"):
            continue
        meta[utt_id] = {
            "speaker": speaker,
            "utt_id": utt_id,
            "gender": gender,
            "codec": "nocodec" if codec == "-" else codec,
            "attack_tag": "bonafide" if attack_tag == "-" else attack_tag,
            "attack": "bonafide" if attack_label == "bonafide" else attack_label,
            "label": 1 if key == "bonafide" else 0,
        }
    print(f"HF protocol rows for {split} Track 1: {len(meta):,}")
    if not meta:
        print("Protocol sample keys:", list(first.keys()))
        inspect_asv5_hf_stream(HF_DEBUG_N)
        raise RuntimeError("ASVspoof 5 HF protocol parsed zero rows. Send the printed protocol/sample keys.")
    return meta


def describe_hf_sample(sample: dict, max_value_chars: int = 120) -> dict:
    """Return a compact printable description of one HF WebDataset sample."""
    desc = {"__key__": sample.get("__key__"), "fields": {}}
    for key, value in sample.items():
        if key == "__key__":
            continue
        if isinstance(value, bytes):
            desc["fields"][key] = f"bytes[{len(value):,}]"
        elif isinstance(value, dict):
            desc["fields"][key] = f"dict keys={list(value.keys())}"
        elif isinstance(value, str):
            compact = value[:max_value_chars].replace("\n", "\\n")
            desc["fields"][key] = f"str[{len(value):,}] {compact}"
        else:
            desc["fields"][key] = type(value).__name__
    return desc


def inspect_asv5_hf_stream(n: int = 30) -> list[dict]:
    """Print the first HF samples so real keys and payload fields are visible."""
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import load_dataset

    stream = load_dataset("jungjee/asvspoof5", split="train", streaming=True)
    rows = []
    print(f"Inspecting first {n} ASVspoof 5 HF stream samples")
    for i, sample in enumerate(stream):
        desc = describe_hf_sample(sample)
        rows.append(desc)
        print(f"[{i}] key={desc['__key__']} fields={desc['fields']}")
        if i + 1 >= n:
            break
    return rows


def hf_audio_ref(sample: dict):
    """Return (utt_id, audio_ref) from one HF WebDataset example when it is audio."""
    key = sample.get("__key__", "")
    if not isinstance(key, str) or "/" not in key:
        return None, None
    utt_id = key.split("/")[-1]
    for audio_key in ["flac", "audio", "wav", "mp3"]:
        if audio_key in sample and isinstance(sample[audio_key], (bytes, dict, str)):
            return utt_id, sample[audio_key]
    for audio_key, value in sample.items():
        if audio_key.startswith("__"):
            continue
        if isinstance(value, (bytes, dict, str)):
            return utt_id, value
    return None, None


def asv5_hf_candidate_ids(sample: dict) -> list[str]:
    """Generate possible protocol utterance IDs from a HF sample."""
    candidates = []
    key = sample.get("__key__", "")
    if isinstance(key, str) and key:
        candidates.extend([key, key.split("/")[-1]])
        last = key.split("/")[-1]
        candidates.extend([Path(last).stem, last.replace(".flac", "")])
    for field, value in sample.items():
        if field.startswith("__"):
            continue
        if isinstance(value, str):
            candidates.extend([value, Path(value).name, Path(value).stem])

    seen = set()
    out = []
    for item in candidates:
        item = str(item)
        if item and item not in seen:
            seen.add(item)
            out.append(item)
    return out


def resolve_asv5_hf_utt_id(sample: dict, protocol: dict) -> str | None:
    """Find the protocol utterance ID matching a HF sample."""
    for candidate in asv5_hf_candidate_ids(sample):
        if candidate in protocol:
            return candidate
    return None


def predict_hf_batch(model, model_name: str, audio_items: list[tuple[str, object, int]], entry: dict):
    inputs, labels, utt_ids = [], [], []
    for utt_id, audio_bytes, label in audio_items:
        wav = load_audio_16k(audio_bytes)
        if entry["dataset"] is WaveformDataset:
            inputs.append(prepare_waveform(wav))
        else:
            inputs.append(prepare_lfcc(wav))
        labels.append(label)
        utt_ids.append(utt_id)
    batch = torch.stack(inputs)
    return entry["predict"](model, batch), labels, utt_ids


def build_asv5_hf_index(protocol: dict) -> pd.DataFrame:
    """Scan the HF stream and verify that audio samples match protocol rows before inference."""
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import load_dataset

    stream = load_dataset("jungjee/asvspoof5", split="train", streaming=True)
    split_letter = asv5_prefix_for_split(ASV5_SPLIT)
    rows = []
    seen_prefixes = {}
    unmatched_examples = []
    scanned = 0
    audio_like = 0

    for sample in tqdm(stream, desc="scan_asv5_hf"):
        scanned += 1
        key = sample.get("__key__", "")
        top_prefix = key.split("/")[0] if isinstance(key, str) and "/" in key else str(key)
        seen_prefixes[top_prefix] = seen_prefixes.get(top_prefix, 0) + 1

        _, audio = hf_audio_ref(sample)
        if audio is None:
            continue
        audio_like += 1

        # Prefer the expected flac_D/flac_E/flac_T prefix, but do not require it:
        # some HF loaders expose only the filename. The protocol match is authoritative.
        if isinstance(key, str) and f"flac_{split_letter}" not in key and "/" in key:
            continue

        utt_id = resolve_asv5_hf_utt_id(sample, protocol)
        if utt_id is None:
            if len(unmatched_examples) < 5:
                unmatched_examples.append(describe_hf_sample(sample))
        else:
            meta = protocol[utt_id]
            rows.append({
                "utt_id": utt_id,
                "label": int(meta["label"]),
                "speaker": meta.get("speaker", "unknown"),
                "codec": meta.get("codec", "unknown"),
                "attack_tag": meta.get("attack_tag", "unknown"),
                "attack": meta.get("attack", "unknown"),
                "hf_key": key,
            })

        if SMOKE_TEST_N is not None and len(rows) >= SMOKE_TEST_N:
            break
        if SMOKE_TEST_N is None and len(rows) >= len(protocol):
            break

    df = pd.DataFrame(rows)
    print("ASVspoof 5 HF scan summary")
    print(f"  scanned samples     : {scanned:,}")
    print(f"  audio-like samples  : {audio_like:,}")
    print(f"  protocol rows       : {len(protocol):,}")
    print(f"  matched audio rows  : {len(df):,}")
    print(f"  seen key prefixes   : {seen_prefixes}")
    if unmatched_examples:
        print("  unmatched audio examples:")
        for example in unmatched_examples:
            print(f"    key={example['__key__']} fields={example['fields']}")
    if df.empty:
        inspect_asv5_hf_stream(HF_DEBUG_N)
        raise RuntimeError(
            "ASVspoof 5 HF scan matched zero audio rows. "
            "Send the printed inspect/scan output so the matcher can be adjusted."
        )
    print(df.head())
    return df


def evaluate_asv5_hf_webdataset_model(
    model_name: str,
    results: dict,
    output_pkl: Path,
    output_dir: Path,
    force_eval: bool = False,
    partial_save_every: int = 5000,
    partial_input_dirs: list[Path] | None = None,
):
    """Evaluate ASVspoof 5 from the HF WebDataset stream. Local extracted files are faster."""
    if model_name in results and not force_eval:
        print(f"{model_name}: already present in results.pkl, skipping")
        return results[model_name]

    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import load_dataset

    protocol = ASV5_HF_PROTOCOL
    hf_index = ASV5_HF_INDEX
    target_ids = set(hf_index["utt_id"].astype(str))
    entry = MODEL_REGISTRY[model_name]
    model = entry["loader"]()

    partial = load_partial(output_dir, model_name, partial_input_dirs)
    scores, labels, utt_ids = [], [], []
    completed = set()
    if partial:
        scores, labels, utt_ids = partial["scores"], partial["labels"], partial["utt_ids"]
        completed = set(utt_ids)
        print(f"{model_name}: resume partial with {len(completed):,} completed utterances")

    stream = load_dataset("jungjee/asvspoof5", split="train", streaming=True)
    pending = []
    last_save = len(utt_ids)
    t0 = time.time()
    try:
        for sample in tqdm(stream, desc=f"{model_name}:hf_stream"):
            _, audio = hf_audio_ref(sample)
            if audio is None:
                continue
            utt_id = resolve_asv5_hf_utt_id(sample, protocol)
            if utt_id is None or utt_id not in target_ids or utt_id in completed:
                continue
            pending.append((utt_id, audio, int(protocol[utt_id]["label"])))
            if len(pending) >= entry["batch_size"]:
                batch_scores, batch_labels, batch_ids = predict_hf_batch(model, model_name, pending, entry)
                scores.extend(float(x) for x in batch_scores)
                labels.extend(int(x) for x in batch_labels)
                utt_ids.extend(str(x) for x in batch_ids)
                completed.update(batch_ids)
                pending = []
            if len(utt_ids) - last_save >= partial_save_every:
                save_partial(output_dir, model_name, scores, labels, utt_ids)
                last_save = len(utt_ids)
            if SMOKE_TEST_N is not None and len(utt_ids) >= SMOKE_TEST_N:
                break
        if pending:
            batch_scores, batch_labels, batch_ids = predict_hf_batch(model, model_name, pending, entry)
            scores.extend(float(x) for x in batch_scores)
            labels.extend(int(x) for x in batch_labels)
            utt_ids.extend(str(x) for x in batch_ids)
    except KeyboardInterrupt:
        save_partial(output_dir, model_name, scores, labels, utt_ids)
        print(f"{model_name}: interrupted; partial saved")
        raise
    finally:
        release_model(model)
        model = None

    if not scores:
        raise RuntimeError("No ASVspoof 5 HF scores were produced. Check ASV5_HF_INDEX and stream matcher output.")

    eer = compute_eer(scores, labels)
    result = {"eer": eer, "scores": np.asarray(scores, dtype=np.float64), "labels": np.asarray(labels, dtype=np.int64)}
    results[model_name] = result
    save_results_pickle(results, output_pkl)
    clear_partial(output_dir, model_name)
    print(f"{model_name}: EER={eer:.4f}% N={len(scores):,} elapsed={(time.time()-t0)/60:.1f} min")
    return result

In [ ]:
preflight_check_model_inputs(ENABLED_MODELS, results=results, force_eval=FORCE_EVAL)

if RESOLVED_ASV5_SOURCE == "local":
    run_eval_plan(
        enabled_models=ENABLED_MODELS,
        df=eval_df,
        results=results,
        output_pkl=OUTPUT_PKL,
        output_dir=OUTPUT_DIR,
        force_eval=FORCE_EVAL,
        partial_save_every=PARTIAL_SAVE_EVERY,
        num_workers=NUM_WORKERS,
        compound_enabled=RUN_COMPOUND_SSL_MODELS,
        partial_input_dirs=PARTIAL_INPUT_DIRS,
    )
elif RESOLVED_ASV5_SOURCE == "hf_tar":
    ASV5_HF_TAR_PROTOCOL = load_asv5_hf_tar_protocol(ASV5_SPLIT)
    ASV5_HF_TAR_INDEX = build_asv5_hf_tar_index(ASV5_HF_TAR_PROTOCOL)
    for model_name in ENABLED_MODELS:
        evaluate_asv5_hf_tar_model(
            model_name=model_name,
            results=results,
            output_pkl=OUTPUT_PKL,
            output_dir=OUTPUT_DIR,
            force_eval=FORCE_EVAL.get(model_name, False),
            partial_save_every=PARTIAL_SAVE_EVERY,
            partial_input_dirs=PARTIAL_INPUT_DIRS,
        )
elif RESOLVED_ASV5_SOURCE == "hf_webdataset":
    ASV5_HF_PROTOCOL = load_asv5_hf_protocol(ASV5_SPLIT)
    inspect_asv5_hf_stream(HF_DEBUG_N)
    ASV5_HF_INDEX = build_asv5_hf_index(ASV5_HF_PROTOCOL)
    for model_name in ENABLED_MODELS:
        evaluate_asv5_hf_webdataset_model(
            model_name=model_name,
            results=results,
            output_pkl=OUTPUT_PKL,
            output_dir=OUTPUT_DIR,
            force_eval=FORCE_EVAL.get(model_name, False),
            partial_save_every=PARTIAL_SAVE_EVERY,
            partial_input_dirs=PARTIAL_INPUT_DIRS,
        )
else:
    raise ValueError(f"Unknown RESOLVED_ASV5_SOURCE={RESOLVED_ASV5_SOURCE}")

save_results_pickle(results, OUTPUT_PKL)
print("final summary")
for name, result in results.items():
    print(f"{name:20s} EER={result['eer']:.4f}% N={len(result['scores']):,}")